In [ ]:
import json
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field

@dataclass
class SchemaRule:
    """Defines validation rules for a schema property"""
    property_name: str
    required: bool = False
    recommended: bool = False
    expected_types: List[str] = field(default_factory=list)
    description: str = ""
    min_length: Optional[int] = None
    max_length: Optional[int] = None
    pattern: Optional[str] = None
    enum_values: Optional[List[str]] = None
    nested_rules: Optional[List['SchemaRule']] = None

class UniversalSchemaValidator:
    """
    Universal Schema.org JSON-LD validator
    Works with ANY schema type (Service, Product, Organization, Event, etc.)
    """

    def __init__(self):
        self.score = 100
        self.recommendations: List[Dict] = []
        self.properties_found: List[str] = []
        self.properties_missing: List[str] = []
        self.checks_performed: List[str] = []

        # Schema.org type definitions with their requirements
        self.schema_definitions = self._load_schema_definitions()

    def _load_schema_definitions(self) -> Dict[str, List[SchemaRule]]:
        """Define validation rules for common Schema.org types"""

        return {
            "Service": [
                SchemaRule("name", required=True, description="Service name"),
                SchemaRule("description", required=True, description="Service description", min_length=50, max_length=300),
                SchemaRule("url", required=True, description="URL to service page"),
                SchemaRule("@id", recommended=True, description="Unique identifier"),
                SchemaRule("provider", recommended=True, description="Organization providing the service"),
                SchemaRule("offers", recommended=True, description="Pricing information",
                          nested_rules=[
                              SchemaRule("@type", required=True, expected_types=["Offer"]),
                              SchemaRule("price", required=True, description="Price amount"),
                              SchemaRule("priceCurrency", required=True, description="Currency code (e.g., INR, USD)"),
                              SchemaRule("availability", recommended=True, expected_types=["InStock", "OutOfStock"])
                          ]),
                SchemaRule("areaServed", recommended=True, description="Geographic area"),
                SchemaRule("category", recommended=True, description="Service category")
            ],
            "Product": [
                SchemaRule("name", required=True),
                SchemaRule("description", required=True, min_length=50, max_length=500),
                SchemaRule("image", required=True),
                SchemaRule("brand", recommended=True),
                SchemaRule("offers", recommended=True,
                          nested_rules=[
                              SchemaRule("price", required=True),
                              SchemaRule("priceCurrency", required=True),
                              SchemaRule("availability", required=True)
                          ]),
                SchemaRule("aggregateRating", recommended=True),
                SchemaRule("review", recommended=True)
            ],
            "Organization": [
                SchemaRule("name", required=True),
                SchemaRule("url", required=True),
                SchemaRule("logo", required=True),
                SchemaRule("description", recommended=True),
                SchemaRule("sameAs", recommended=True, description="Social media profiles"),
                SchemaRule("contactPoint", recommended=True)
            ],
            "Person": [
                SchemaRule("name", required=True),
                SchemaRule("jobTitle", recommended=True),
                SchemaRule("worksFor", recommended=True),
                SchemaRule("sameAs", recommended=True)
            ],
            "Event": [
                SchemaRule("name", required=True),
                SchemaRule("startDate", required=True),
                SchemaRule("location", required=True),
                SchemaRule("description", recommended=True),
                SchemaRule("offers", recommended=True)
            ],
            "ItemList": [
                SchemaRule("itemListElement", required=True),
                SchemaRule("numberOfItems", recommended=True)
            ],
            "ListItem": [
                SchemaRule("position", required=True),
                SchemaRule("item", required=True)
            ],
            "Offer": [
                SchemaRule("price", required=True),
                SchemaRule("priceCurrency", required=True),
                SchemaRule("availability", recommended=True),
                SchemaRule("validFrom", recommended=True)
            ],
            "BreadcrumbList": [
                SchemaRule("itemListElement", required=True)
            ],
            "FAQPage": [
                SchemaRule("mainEntity", required=True)
            ],
            "Question": [
                SchemaRule("name", required=True),
                SchemaRule("acceptedAnswer", required=True)
            ],
            "Answer": [
                SchemaRule("text", required=True)
            ],
            "Article": [
                SchemaRule("headline", required=True),
                SchemaRule("author", required=True),
                SchemaRule("datePublished", required=True),
                SchemaRule("image", required=True),
                SchemaRule("publisher", required=True)
            ],
            "WebPage": [
                SchemaRule("name", required=True),
                SchemaRule("description", recommended=True),
                SchemaRule("url", recommended=True),
                SchemaRule("mainEntity", recommended=True)
            ]
        }

    def validate(self, schema_json: Dict) -> Dict[str, Any]:
        """
        Main validation method - works with ANY Schema.org JSON

        Args:
            schema_json: Your Schema.org JSON-LD object

        Returns:
            Validation report with score and recommendations
        """
        self.score = 100
        self.recommendations = []
        self.properties_found = []
        self.properties_missing = []
        self.checks_performed = []

        # Step 1: Basic structure validation
        self._validate_basic_structure(schema_json)

        # Step 2: Detect schema type and validate
        schema_type = schema_json.get("@type", "Unknown")
        self.checks_performed.append(f"Detected schema type: {schema_type}")

        # Step 3: Validate based on type
        self._validate_by_type(schema_json, schema_type, path="root")

        # Step 4: Check for nested objects and validate them
        self._validate_nested_objects(schema_json, path="root")

        # Step 5: Calculate final score and generate report
        return self._generate_report(schema_json)

    def _validate_basic_structure(self, data: Dict):
        """Validate @context and @type exist"""

        # Check @context
        if "@context" not in data:
            self._add_recommendation(
                priority="CRITICAL",
                category="Structure",
                issue="Missing @context",
                current="Not found",
                fix="Add '@context': 'https://schema.org'",
                impact="Search engines cannot parse your schema without this",
                penalty=25
            )
            self.properties_missing.append("@context")
        else:
            self.properties_found.append("@context")
            if data["@context"] != "https://schema.org":
                self._add_recommendation(
                    priority="HIGH",
                    category="Structure",
                    issue="Incorrect @context value",
                    current=data["@context"],
                    fix="Change to 'https://schema.org'",
                    impact="Must be exact URL for proper parsing",
                    penalty=10
                )

        # Check @type
        if "@type" not in data:
            self._add_recommendation(
                priority="CRITICAL",
                category="Structure",
                issue="Missing @type",
                current="Not found",
                fix="Add '@type': 'Service' (or appropriate type)",
                impact="Required to identify what schema type this is",
                penalty=25
            )
            self.properties_missing.append("@type")
        else:
            self.properties_found.append(f"@type: {data['@type']}")

        self.checks_performed.append("Basic structure validation (@context, @type)")

    def _validate_by_type(self, data: Dict, schema_type: str, path: str = ""):
        """Validate properties based on schema type definition"""

        if schema_type not in self.schema_definitions:
            # Unknown type - do generic validation
            self.checks_performed.append(f"No specific rules for type: {schema_type}, using generic validation")
            self._validate_generic_object(data, path)
            return

        rules = self.schema_definitions[schema_type]

        for rule in rules:
            full_path = f"{path}.{rule.property_name}" if path != "root" else rule.property_name

            if rule.property_name not in data:
                if rule.required:
                    self._add_recommendation(
                        priority="HIGH",
                        category=f"{schema_type} Properties",
                        issue=f"Missing required property: {rule.property_name}",
                        current="Not found",
                        fix=f"Add '{rule.property_name}'",
                        impact=rule.description or f"Required for {schema_type} schema",
                        penalty=10
                    )
                    self.properties_missing.append(full_path)
                elif rule.recommended:
                    self._add_recommendation(
                        priority="MEDIUM",
                        category=f"{schema_type} Properties",
                        issue=f"Missing recommended property: {rule.property_name}",
                        current="Not found",
                        fix=f"Add '{rule.property_name}'",
                        impact=rule.description or f"Recommended for better SEO",
                        penalty=5
                    )
                    self.properties_missing.append(full_path)
            else:
                self.properties_found.append(full_path)
                value = data[rule.property_name]

                # Validate nested rules if present
                if rule.nested_rules and isinstance(value, dict):
                    for nested_rule in rule.nested_rules:
                        self._validate_nested_property(
                            value, nested_rule, full_path, schema_type
                        )

                # Validate array items
                elif rule.nested_rules and isinstance(value, list):
                    for i, item in enumerate(value):
                        if isinstance(item, dict):
                            for nested_rule in rule.nested_rules:
                                self._validate_nested_property(
                                    item, nested_rule, f"{full_path}[{i}]", schema_type
                                )

                # Check string length constraints
                elif isinstance(value, str):
                    if rule.min_length and len(value) < rule.min_length:
                        self._add_recommendation(
                            priority="MEDIUM",
                            category=f"{schema_type} Content",
                            issue=f"{rule.property_name} too short",
                            current=f"{len(value)} characters",
                            fix=f"Extend to at least {rule.min_length} characters",
                            impact="Better for SEO and user understanding",
                            penalty=3
                        )

                    if rule.max_length and len(value) > rule.max_length:
                        self._add_recommendation(
                            priority="LOW",
                            category=f"{schema_type} Content",
                            issue=f"{rule.property_name} too long",
                            current=f"{len(value)} characters",
                            fix=f"Reduce to max {rule.max_length} characters",
                            impact="May be truncated in search results",
                            penalty=2
                        )

        self.checks_performed.append(f"Validated {schema_type} properties")

    def _validate_nested_property(self, parent_data: Dict, rule: SchemaRule, parent_path: str, parent_type: str):
        """Validate a nested property within an object"""

        full_path = f"{parent_path}.{rule.property_name}"

        if rule.property_name not in parent_data:
            if rule.required:
                self._add_recommendation(
                    priority="HIGH",
                    category=f"{parent_type} Nested Property",
                    issue=f"Missing required: {full_path}",
                    current="Not found",
                    fix=f"Add '{rule.property_name}'",
                    impact=rule.description or "Required for complete data",
                    penalty=8
                )
                self.properties_missing.append(full_path)
            elif rule.recommended:
                self._add_recommendation(
                    priority="MEDIUM",
                    category=f"{parent_type} Nested Property",
                    issue=f"Missing recommended: {full_path}",
                    current="Not found",
                    fix=f"Add '{rule.property_name}'",
                    impact=rule.description or "Recommended for better results",
                    penalty=4
                )
                self.properties_missing.append(full_path)
        else:
            self.properties_found.append(full_path)
            value = parent_data[rule.property_name]

            # Check expected types
            if rule.expected_types and value not in rule.expected_types:
                self._add_recommendation(
                    priority="MEDIUM",
                    category=f"{parent_type} Type Check",
                    issue=f"Unexpected value for {full_path}",
                    current=f"{value}",
                    fix=f"Use one of: {', '.join(rule.expected_types)}",
                    impact="Should match Schema.org expected values",
                    penalty=3
                )

    def _validate_nested_objects(self, data: Dict, path: str = ""):
        """Recursively validate nested @type objects"""

        for key, value in data.items():
            if isinstance(value, dict) and "@type" in value:
                nested_type = value["@type"]
                nested_path = f"{path}.{key}" if path != "root" else key
                self._validate_by_type(value, nested_type, nested_path)
                self._validate_nested_objects(value, nested_path)

            elif isinstance(value, list):
                for i, item in enumerate(value):
                    if isinstance(item, dict) and "@type" in item:
                        nested_type = item["@type"]
                        nested_path = f"{path}.{key}[{i}]" if path != "root" else f"{key}[{i}]"
                        self._validate_by_type(item, nested_type, nested_path)
                        self._validate_nested_objects(item, nested_path)

    def _validate_generic_object(self, data: Dict, path: str):
        """Generic validation for unknown schema types"""

        # Check for common recommended properties
        common_props = ["name", "description", "url", "image", "@id"]

        for prop in common_props:
            if prop not in data:
                self._add_recommendation(
                    priority="LOW",
                    category="Generic Validation",
                    issue=f"Missing common property: {prop}",
                    current="Not found",
                    fix=f"Consider adding '{prop}'",
                    impact="Generally recommended for all schema types",
                    penalty=2
                )
                self.properties_missing.append(f"{path}.{prop}")
            else:
                self.properties_found.append(f"{path}.{prop}")

    def _add_recommendation(self, priority: str, category: str, issue: str,
                           current: str, fix: str, impact: str, penalty: int):
        """Add a recommendation and deduct score"""

        self.recommendations.append({
            "priority": priority,
            "category": category,
            "issue": issue,
            "current": current,
            "fix": fix,
            "impact": impact
        })
        self.score -= penalty

    def _generate_report(self, original_data: Dict) -> Dict[str, Any]:
        """Generate final validation report"""

        final_score = max(0, min(100, self.score))

        # Sort recommendations by priority
        priority_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
        sorted_recommendations = sorted(
            self.recommendations,
            key=lambda x: priority_order.get(x["priority"], 4)
        )

        # Count by priority
        critical = len([r for r in self.recommendations if r["priority"] == "CRITICAL"])
        high = len([r for r in self.recommendations if r["priority"] == "HIGH"])
        medium = len([r for r in self.recommendations if r["priority"] == "MEDIUM"])
        low = len([r for r in self.recommendations if r["priority"] == "LOW"])

        # Determine grade
        if final_score >= 90:
            grade, status = "A", "Excellent"
        elif final_score >= 80:
            grade, status = "B", "Good"
        elif final_score >= 70:
            grade, status = "C", "Needs Improvement"
        elif final_score >= 60:
            grade, status = "D", "Poor"
        else:
            grade, status = "F", "Critical Issues"

        return {
            "score": final_score,
            "grade": grade,
            "status": status,
            "schema_type": original_data.get("@type", "Unknown"),
            "summary": {
                "total_issues": len(self.recommendations),
                "critical": critical,
                "high": high,
                "medium": medium,
                "low": low,
                "properties_found": len(self.properties_found),
                "properties_missing": len(self.properties_missing)
            },
            "recommendations": sorted_recommendations,
            "properties": {
                "found": self.properties_found,
                "missing": self.properties_missing
            },
            "checks_performed": self.checks_performed,
            "validated_at": "2026-03-17T12:05:00"
        }


def print_report(report: Dict):
    """Print formatted validation report"""

    print("=" * 70)
    print(f"SCHEMA.ORG VALIDATION REPORT")
    print("=" * 70)
    print(f"Schema Type: {report['schema_type']}")
    print(f"Score: {report['score']}/100")
    print(f"Grade: {report['grade']} ({report['status']})")
    print("-" * 70)
    print(f"Total Issues: {report['summary']['total_issues']}")
    print(f"  🔴 Critical: {report['summary']['critical']}")
    print(f"  🟠 High: {report['summary']['high']}")
    print(f"  🟡 Medium: {report['summary']['medium']}")
    print(f"  🟢 Low: {report['summary']['low']}")
    print(f"\nProperties Found: {report['summary']['properties_found']}")
    print(f"Properties Missing: {report['summary']['properties_missing']}")
    print("=" * 70)

    if report['recommendations']:
        print("\nRECOMMENDATIONS (by priority):")
        print("-" * 70)

        for i, rec in enumerate(report['recommendations'], 1):
            icon = "🔴" if rec['priority'] == "CRITICAL" else \
                   "🟠" if rec['priority'] == "HIGH" else \
                   "🟡" if rec['priority'] == "MEDIUM" else "🟢"
            print(f"\n{i}. {icon} [{rec['priority']}] {rec['category']}")
            print(f"   Issue: {rec['issue']}")
            print(f"   Current: {rec['current']}")
            print(f"   Fix: {rec['fix']}")
            print(f"   Impact: {rec['impact']}")
    else:
        print("\n✅ No issues found! Your schema is perfect.")

    print("\n" + "=" * 70)
    print("CHECKS PERFORMED:")
    print("-" * 70)
    for check in report['checks_performed']:
        print(f"  ✓ {check}")
    print("=" * 70)


# ============ USAGE EXAMPLES ============

if __name__ == "__main__":
    validator = UniversalSchemaValidator()

    # Example 1: Your original ItemList
    print("\n" + "=" * 70)
    print("EXAMPLE 1: ItemList (Your Original JSON)")
    print("=" * 70)

    itemlist_json = {
  "@context": "https://schema.org",
  "@type": "BreadcrumbList",
  "itemListElement": [
    {
      "@type": "ListItem",
      "position": 1,
      "name": "Home",
      "item": "https://primecounsel.in/"
    },
    {
      "@type": "ListItem",
      "position": 2,
      "name": "Services",
      "item": "https://primecounsel.in/services"
    },
    {
      "@type": "ListItem",
      "position": 3,
      "name": "CURRENT_PAGE_NAME",
      "item": "https://primecounsel.in/services/CURRENT_PAGE_URL"
    }
  ]
}

    report1 = validator.validate(itemlist_json)
    print_report(report1)

    # Example 2: Product schema
    print("\n" + "=" * 70)
    print("EXAMPLE 2: Product Schema")
    print("=" * 70)

    product_json = {
        "@context": "https://schema.org",
        "@type": "Product",
        "name": "Wireless Headphones",
        "description": "Noise cancelling bluetooth headphones",
        "image": "https://example.com/image.jpg",
        "offers": {
            "@type": "Offer",
            "price": "99.99",
            "priceCurrency": "USD"
        }
    }

    report2 = validator.validate(product_json)
    print_report(report2)

    # Example 3: Organization schema
    print("\n" + "=" * 70)
    print("EXAMPLE 3: Organization Schema")
    print("=" * 70)

    org_json = {
        "@context": "https://schema.org",
        "@type": "Organization",
        "name": "Tech Corp",
        "url": "https://techcorp.com"
    }

    report3 = validator.validate(org_json)
    print_report(report3)

    # Example 4: Custom/Unknown schema type
    print("\n" + "=" * 70)
    print("EXAMPLE 4: Custom Schema Type")
    print("=" * 70)

    custom_json = {
        "@context": "https://schema.org",
        "@type": "CustomType",
        "customField": "value"
    }

    report4 = validator.validate(custom_json)
    print_report(report4)

Get Schema


In [ ]:
import requests
import re
import json
from urllib.parse import urljoin

def extract_schema_advanced(url):
    """Advanced schema extraction with multiple methods"""

    # Try different user agents
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.0',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.0',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.0'
    ]

    schemas = []

    for i, ua in enumerate(user_agents):
        try:
            print(f"\n🔄 Attempt {i+1} with different headers...")

            headers = {
                'User-Agent': ua,
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                'Accept-Language': 'en-US,en;q=0.5',
                'Accept-Encoding': 'gzip, deflate, br',
                'Connection': 'keep-alive',
                'Upgrade-Insecure-Requests': '1',
                'Sec-Fetch-Dest': 'document',
                'Sec-Fetch-Mode': 'navigate',
                'Sec-Fetch-Site': 'none',
                'Cache-Control': 'max-age=0'
            }

            session = requests.Session()
            # First get the page
            response = session.get(url, headers=headers, timeout=30, allow_redirects=True)
            print(f"   Status: {response.status_code}")
            print(f"   URL: {response.url}")

            if response.status_code == 200:
                html = response.text
                print(f"   Page size: {len(html)} characters")

                # Method 1: Standard JSON-LD
                pattern1 = r'<script[^>]*type=["\']application/ld\+json["\'][^>]*>(.*?)</script>'
                matches = re.findall(pattern1, html, re.DOTALL | re.IGNORECASE)
                print(f"   Found {len(matches)} potential JSON-LD scripts")

                for j, match in enumerate(matches):
                    try:
                        clean = match.strip()
                        if not clean:
                            continue
                        data = json.loads(clean)
                        schemas.append({
                            "method": "json-ld",
                            "index": len(schemas),
                            "type": data.get("@type", "Unknown"),
                            "context": data.get("@context", "None"),
                            "data": data
                        })
                        print(f"   ✅ Parsed schema {len(schemas)}: {data.get('@type', 'Unknown')}")
                    except Exception as e:
                        print(f"   ❌ Failed to parse script {j}: {str(e)[:50]}")

                # Method 2: Microdata (itemscope/itemtype)
                microdata_pattern = r'<[^>]*itemtype=["\']([^"\']+)["\'][^>]*>'
                microdata_matches = re.findall(microdata_pattern, html, re.IGNORECASE)
                if microdata_matches:
                    print(f"   Found {len(microdata_matches)} microdata types: {set(microdata_matches)}")

                # Method 3: RDFa
                rdfa_pattern = r'<[^>]*typeof=["\']([^"\']+)["\'][^>]*>'
                rdfa_matches = re.findall(rdfa_pattern, html, re.IGNORECASE)
                if rdfa_matches:
                    print(f"   Found {len(rdfa_matches)} RDFa types: {set(rdfa_matches)}")

                # Method 4: Look for schema in data attributes
                data_schema_pattern = r'data-schema=["\']([^"\']+)["\']'
                data_matches = re.findall(data_schema_pattern, html, re.IGNORECASE)
                if data_matches:
                    print(f"   Found {len(data_matches)} data-schema attributes")

                if schemas:
                    break  # Success, no need to try other user agents

        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue

    # Save results
    result = {
        "url": url,
        "schemas_found": len(schemas),
        "schemas": schemas,
        "timestamp": "2026-03-17T13:37:00"
    }

    with open("primecounsel_schemas.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n{'='*60}")
    print(f"RESULT: {len(schemas)} schema(s) found")
    print(f"Saved to: primecounsel_schemas.json")
    print(f"{'='*60}")

    return result

# Run it
url = "https://primecounsel.in/"
result = extract_schema_advanced(url)

# If still no schemas, try to get page content for manual inspection
if result["schemas_found"] == 0:
    print("\n🔍 No schemas found. Trying to fetch page for manual inspection...")
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        print(f"\nFirst 2000 characters of HTML:")
        print(response.text[:2000])
        print("\n...[truncated]...")

        # Check for common issues
        text = response.text.lower()
        if "cloudflare" in text:
            print("\n⚠️  Cloudflare protection detected")
        if "captcha" in text:
            print("\n⚠️  CAPTCHA detected")
        if "access denied" in text:
            print("\n⚠️  Access denied")

    except Exception as e:
        print(f"Could not fetch page: {e}")

**Extracting Schema.Org from the page**

Run this in developer mode

In [ ]:
// Extract all Schema.org data from the page
const results = {
    jsonld: [],
    microdata: [],
    rdfa: []
};

// Method 1: JSON-LD (most common)
document.querySelectorAll('script[type="application/ld+json"]').forEach((script, i) => {
    try {
        const data = JSON.parse(script.innerText);
        results.jsonld.push({
            index: i,
            type: data['@type'] || 'Unknown',
            data: data
        });
        console.log(`✅ JSON-LD ${i}: ${data['@type'] || 'Unknown'}`);
    } catch(e) {
        console.log(`❌ JSON-LD ${i} invalid:`, e.message);
    }
});

// Method 2: Microdata
document.querySelectorAll('[itemscope]').forEach((el, i) => {
    const type = el.getAttribute('itemtype');
    results.microdata.push({
        index: i,
        type: type || 'Unknown'
    });
    console.log(`📦 Microdata ${i}: ${type || 'Unknown'}`);
});

// Method 3: RDFa
document.querySelectorAll('[typeof]').forEach((el, i) => {
    const type = el.getAttribute('typeof');
    results.rdfa.push({
        index: i,
        type: type
    });
    console.log(`🏷️ RDFa ${i}: ${type}`);
});

// Summary
console.log('\n' + '='.repeat(50));
console.log('SCHEMA SUMMARY:');
console.log(`JSON-LD: ${results.jsonld.length} found`);
console.log(`Microdata: ${results.microdata.length} found`);
console.log(`RDFa: ${results.rdfa.length} found`);
console.log('='.repeat(50));

// Copy to clipboard
copy(JSON.stringify(results, null, 2));
console.log('\n✅ Copied to clipboard! Paste it in the chat.');